# Controlled radial distance fitting: a separate side experiment

Select **GPU → Run all**. Existing ranking results are preserved. No planning is used.

We fit the adapted latent MSE to **Euclidean distance / 34**:

`loss = mean((adapted_MSE - physical_distance / 34)**2)`

The target is distance itself, not squared distance. Multiplying adapted MSE by34 gives predicted distance in simulation units. The scale stays34 at **every stage**; predictions are never clipped.

Same fixed goal **(50,42)**, layout,24training rays every15°, radii2,6,...34 and cumulative stages **6 → 14 → 22 → 34**. Start a fresh identity adapter; later fits keep earlier positions and continue its weights. Frozen encoder, no ranking term, no anchor loss.

Each stage uses500updates with32uniformly sampled positions/update, matching32observation evaluations in the old16pair batches. Sampling distributions differ, so this compares fitting recipes rather than an identical-exposure objective-only ablation.

New timestamped results go under `radial_distance_v1`; no old adapter or result folder is required.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_distance_v1' / RUN_ID
CACHE = Path('/content/radial-distance-cache') / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-distance')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = 'e22cd39a8945f56e2691f656713d753c37ada1ae'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-distance-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Local feature cache:', CACHE)


## Frozen encoder and renderer

Only the existing I-JEPA encoder checkpoint is needed. Old adapter weights, experience archives and world-model checkpoints are not used.


In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
assert shutil.disk_usage('/content').free > 7*2**30, 'At least 7 GiB free local disk is needed for the float32 feature cache.'
print('Ready. No world-model checkpoint or previous experiment folder is required.')

def stage_command(operation,index=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_experiment',operation,
          '--output-dir',OUTPUT,'--cache-dir',CACHE,'--device','cuda']
    if operation=='prepare': args+=['--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--objective','distance']
    if index is not None: args+=['--stage-index',str(index)]
    return args


## Encode all fixed positions once

About 4,668 unique images are freshly rendered and encoded in batches of eight. The **float32 probe cache is about5.7GiB and stays on local Colab disk**, avoiding repeated encoder runs. Compact results, training features and the fresh goal image/tokens are saved in Drive.

Pre-encoding a position does not make it a training input: each stage's optimizer receives only its explicitly allowed radial training positions. Midpoints and withheld directions never enter fitting. Farther training positions enter fitting only when their stage is reached.

The encoder is the main cost of this notebook. It runs once for all four stages.


In [ ]:
started=time.monotonic()
shell(stage_command('prepare'),cwd=REPO)
design=json.loads((OUTPUT/'design.json').read_text())
config=json.loads((OUTPUT/'config.json').read_text())
print(f'Fresh encoding took {(time.monotonic()-started)/60:.1f} minutes.')
print('Unique encoded images:',len(design['points']))
print('Stage limits and cumulative training positions:',[(s['outer_radius'],len(s['training_ids'])) for s in design['stages']])
print('Matched offset families:',len(design['matched_family_ids']))
assert config['kind']=='radial_distance_v1' and config['distance_scale']==34
print('Fitting:',config['fitting'],'| squared error to distance /34; no ranking or anchor loss')
print('Fresh goal image:')
from IPython.display import display,Image
display(Image(filename=str(OUTPUT/'goal.png')))


## Four cumulative distance fits and generalization checks

Only positions inside the current approved training set enter the optimizer. Later stages retain all previous positions.

**Read both kinds of evidence:**
- Distance MAE/RMSE measure how accurately the cost estimates physical distance; smaller is better.
- Uphill fractions measure how often moving toward the goal raises the cost; smaller is better.
- Full training regression loss is separate from noisy minibatch losses.
- The distance-calibration figure compares predicted distance to the true diagonal. Raw MSE is calibrated by one nonnegative scalar fitted on the current training positions only. Beating that baseline indicates more than a change of units.
- Heatmaps cover the full right compartment with independent, unclipped raw/adapted scales. Cost-versus-radius plots should rise; interpolation plots toward the goal should fall.
- `inside` and `outside` directional step comparisons classify both endpoints; `crossing` is separate.
- Distance errors use deduplicated, genuinely heldout directional positions, excluding coordinate and rendered-image overlaps with training inputs.
- Offset tests use matched families, averaging ± signs within each family. Flat curves are reported separately.
- The **fixed unseen-ray R≤6** distance test stays the same at every stage, so later updates can be checked for damage to earlier generalization.

Rerunning this stage cell in the same runtime reuses validated completed stages. Setup creates a fresh run; a runtime reset loses local tokens but retains Drive results. Stage4 has no directional probes beyond34, so that extrapolation result is N/A.


In [ ]:
def pct(value): return 'N/A' if value is None else f'{value:.1%}'

def show_report(report):
    t=report['training']
    print(f"\nSTAGE {report['stage']} | max radius {report['outer_radius']} | {report['training_positions']} training positions")
    print('Initialization:',report['initialization'],'| point samples:',sum(t['sample_counts']))
    print('Full-training regression loss:',t['full_regression_loss_before'],'→',t['full_regression_loss_after'])
    print('Training distance MAE:',t['distance_mae_before'],'→',t['distance_mae_after'])
    print('Training distance RMSE:',t['distance_rmse_before'],'→',t['distance_rmse_after'])
    d=report['distance']
    print('Target: MSE = distance /34. Raw baseline predicts distance = raw MSE ×',d['raw_calibration'])
    print('\nDISTANCE ERRORS (simulation units; not percentages)')
    print(f"{'cohort':>25} {'positions':>10} {'method':>16} {'MAE':>10} {'RMSE':>10}")
    groups=[('training',d['training']),('heldout inside',d['directional']['inside']),
            ('heldout outside',d['directional']['outside']),('unseen rays inside',d['withheld_direction']['inside']),
            ('unseen rays outside',d['withheld_direction']['outside']),('fixed unseen R<=6',d['fixed_near_withheld_direction'])]
    for label,g in groups:
        for method in ('raw_calibrated','adapted'):
            mae='N/A' if g[method]['mae'] is None else f"{g[method]['mae']:.4f}"
            rmse='N/A' if g[method]['rmse'] is None else f"{g[method]['rmse']:.4f}"
            print(f"{label:>25} {g['count']:>10} {method:>16} {mae:>10} {rmse:>10}")
    print('Raw/adapted goal self-cost:',report['raw_goal_self_cost'],report['adapted_goal_self_cost'])
    print('New primitive actions:',report['primitive_interactions'])
    print('\nDirectional comparisons: uphill means wrong direction while moving TOWARD the goal')
    print(f"{'directions':>20} {'region':>10} {'method':>8} {'steps':>7} {'uphill':>9} {'heldout up / steps':>22} {'ties':>7}")
    for kind,regions in report['generalization'].items():
        for region,methods in regions.items():
            for method,m in methods.items():
                print(f"{kind:>20} {region:>10} {method:>8} {m['comparisons']:>7} {pct(m['up_fraction']):>9} "
                      f"{m['heldout_up_steps']:>9}/{m['heldout_comparisons']:<11} {m['tie_steps']:>7}")
    for key,label in [('offset_fitted_centers','OFFERED TRAINING ENDPOINTS'),('offset_summary','FIXED COHORT ACROSS ALL RADII')]:
        print('\nMATCHED OFFSET TEST:',label)
        print(f"{'offset':>7} {'families':>9} {'method':>8} {'endpoint OK':>12} {'monotone':>10} {'uphill':>9} {'flat':>8}")
        for row in report[key]:
            for method in ('raw','adapted'):
                m=row[method]
                print(f"{row['offset']:>7g} {row['families']:>9} {method:>8} {pct(m['endpoint_correct']):>12} "
                      f"{pct(m['informative_monotone']):>10} {pct(m['up_fraction']):>9} {pct(m['flat']):>8}")
    unseen=sum(report['exposure']['heldout_from_offered_training'])
    print(f'Coordinates with no coordinate/image overlap with offered training: {unseen}/{len(design["points"])}')
    print('Nearest-training distances and per-profile exposure flags are saved in report.json.')

reports=[]
for index in (1,2,3,4):
    started=time.monotonic()
    shell(stage_command('stage',index),cwd=REPO)
    folder=OUTPUT/f'stages/s{index}'
    report=json.loads((folder/'report.json').read_text()); reports.append(report)
    show_report(report)
    print(f'Stage took {(time.monotonic()-started)/60:.1f} minutes.')
    for name in ('heatmaps','distance_calibration','radial_profiles','offset_summary','interpolation_examples','training_loss'):
        display(Image(filename=str(folder/'figures'/f'{name}.png')))


## Compare the four stages

Use heldout directions to assess whether learning extends between training rays. Use the beyond-radius results to see whether it extrapolates before those distances enter training. Compare raw/adapted errors with their denominators, not training loss alone.


In [ ]:
def number(value): return 'N/A' if value is None else f'{value:.3f}'

print('SUMMARY: distance MAE on genuinely unseen directions (simulation units)')
print(f"{'stage':>6} {'R':>5} {'inside raw':>12} {'inside adapt':>13} {'outside raw':>12} {'outside adapt':>14}")
for r in reports:
    d=r['distance']['withheld_direction']
    print(f"{r['stage']:>6} {r['outer_radius']:>5} "
          f"{number(d['inside']['raw_calibrated']['mae']):>12} {number(d['inside']['adapted']['mae']):>13} "
          f"{number(d['outside']['raw_calibrated']['mae']):>12} {number(d['outside']['adapted']['mae']):>14}")
print('\nFIXED NEAR REGION: same genuinely unseen directions at R<=6 after every update')
print(f"{'stage':>6} {'positions':>10} {'raw MAE':>12} {'adapt MAE':>12} {'adapt RMSE':>12}")
for r in reports:
    g=r['distance']['fixed_near_withheld_direction']
    print(f"{r['stage']:>6} {g['count']:>10} {number(g['raw_calibrated']['mae']):>12} "
          f"{number(g['adapted']['mae']):>12} {number(g['adapted']['rmse']):>12}")
print('Raw calibration changes by stage using training inputs only; fixed-near adapted errors compare the same positions.')
print()
print('SUMMARY: uphill fractions on genuinely withheld-direction comparisons')
print(f"{'stage':>6} {'R':>5} {'train':>7} {'inside raw':>12} {'inside adapt':>13} {'outside raw':>12} {'outside adapt':>14}")
for r in reports:
    g=r['generalization']['withheld_direction']
    print(f"{r['stage']:>6} {r['outer_radius']:>5} {r['training_positions']:>7} "
          f"{pct(g['inside']['raw']['heldout_up_fraction']):>12} {pct(g['inside']['adapted']['heldout_up_fraction']):>13} "
          f"{pct(g['outside']['raw']['heldout_up_fraction']):>12} {pct(g['outside']['adapted']['heldout_up_fraction']):>14}")
print('\nInside/outside populations change as the training radius expands; counts are in the detailed reports.')
print('No outside-ray comparisons remain at R34 because the fixed directional probes stop at radius34; N/A is not zero error.')
print('All results and figures:',OUTPUT)


## Return the results

Download the executed notebook (**File → Download → Download .ipynb**) and attach it here. Each stage saves its adapter, fitting metadata, numeric report and six PNG/PDF figures in the new Drive folder.

This is fixed-goal spatial distance supervision. A good fit on training points alone does not establish interpolation, extrapolation, other-goal transfer or planning improvement. Local synthetic checks verify the workflow; GPU/Colab behavior and scientific results are established by this run.
